This notebook will contain no serious model training. Its purpose is to lock down the experiment so we do not accidently introduce leakage or make methodological decisions halfway through modelling.

We complete the following steps:
| Step | What you need to decide/do  | Output                                                        |
| ---- | --------------------------- | ------------------------------------------------------------- |
| 1    | Define prediction timing    | Clear statement of what information exists at prediction time |
| 2    | Define targets              | Regression + classification                                   |
| 3    | Define modelling features   | Final initial `X` columns                                     |
| 4    | Create chronological splits | Train / validation / test                                     |
| 5    | Calculate trivial baselines | Numbers ML must beat                                          |


# Load Data

In [216]:
TABLE = "price_features"
CALENDAR = "no_calendar"

In [217]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [218]:
import pandas as pd
import numpy as np
import duckdb

from src.config import (
    DATABASE_PATH,
    ROLLING_WINDOWS,
    LAGGED_WINDOWS,
    MA_WINDOWS,
    TICKERS,
    START_DATE,
    END_DATE
)

from src.features import (
    build_price_feature_columns
)

CALENDAR = "no_calendar"


def pull_calendar_table(table_name: str, calendar: str) -> pd.DataFrame:

    print("[START] Connecting to database...")

    con = duckdb.connect(DATABASE_PATH)

    print(f"[START] Reading {table_name}")

    df = con.sql(f"""
    
        SELECT *
        FROM {table_name}

    """).df()

    print(f"[DONE] Read {table_name}")

    con.close()

    print(f"[START] Building {CALENDAR} required columns...")

    base_cols = [
        "ticker",
        "date",
        "open",
        "high",
        "low",
        "close",
        "adj_close",
        "volume"
    ]

    feature_columns = build_price_feature_columns(
        calendars = [CALENDAR],
        rolling_windows = ROLLING_WINDOWS,
        lagged_windows = LAGGED_WINDOWS,
        ma_windows = MA_WINDOWS
    )

    expected_columns = base_cols + list(feature_columns.keys())

    calendar_df = df[expected_columns].copy()

    print(f"[DONE] Filtered to {CALENDAR}")

    calendar_df["date"] = pd.to_datetime(calendar_df["date"])

    calendar_df = calendar_df.sort_values(["ticker", "date"])

    print(f"[DONE] Prepared and sorted table")

    return calendar_df

In [219]:
df = pull_calendar_table(
    table_name = TABLE,
    calendar = CALENDAR
)

[START] Connecting to database...
[START] Reading price_features
[DONE] Read price_features
[START] Building no_calendar required columns...
[DONE] Filtered to no_calendar
[DONE] Prepared and sorted table


In [220]:
df.columns

Index(['ticker', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume',
       'daily_return_no_calendar', 'log_return_no_calendar',
       'cumulative_returns_no_calendar', 'rolling_7d_return_no_calendar',
       'rolling_30d_return_no_calendar', 'lag_1_return_no_calendar',
       'lag_5_return_no_calendar', 'moving_avg_20_no_calendar',
       'moving_avg_50_no_calendar', 'price_vs_ma20_no_calendar',
       'relative_volume_no_calendar', 'rolling_30d_volatility_no_calendar',
       'drawdown_no_calendar', 'target_next_day_return_no_calendar',
       'target_direction_no_calendar'],
      dtype='str')

In [221]:
table_count = 0
graph_count = 0
matrix_count = 0
figure_count = 0

# Define Prediction Timing

We make predictions based on data that is only available at the end of the trading day and make a prediction for the next day.

In [222]:
df_1 = df.copy()

ticker = "ticker"
daily_return = "daily_return_no_calendar" 
log_return = "log_return_no_calendar"
target_return = "target_next_day_return_no_calendar"
target_direction = "target_direction_no_calendar" 

columns =[ticker, daily_return, log_return, target_return, target_direction]

df_1 = df_1[columns]

summary_rows = []

for asset, group in df_1.groupby(ticker):

    expected_simple_direction = (group[daily_return].shift(-1) > 0).astype(int)
    expected_log_direction = (group[log_return].shift(-1) > 0).astype(int)

    summary_row = pd.DataFrame({
        "Ticker": asset,

        "Simple Return Match": [
            (group[daily_return].shift(-1).iloc[:-1] == group[target_return].iloc[:-1]).all()
        ],

        "Log Return Match": [
            (group[log_return].shift(-1).iloc[:-1] == group[target_return].iloc[:-1]).all()
        ],

        "Simple Direction Match": [
            (group[target_direction].iloc[:-1] == expected_simple_direction.iloc[:-1]).all()
        ],

        "Log Direction Match": [
            (group[target_direction].iloc[:-1] == expected_log_direction.iloc[:-1]).all()
        ]
    })

    summary_rows.append(summary_row)

summary_table = pd.concat(summary_rows)

display(summary_table.style.hide(axis="index"))


Ticker,Simple Return Match,Log Return Match,Simple Direction Match,Log Direction Match
GLD,True,False,True,True
MU,True,False,True,True
NKE,True,False,True,True
RPI.L,True,False,True,True
SNDK,True,False,True,True
SPY,True,False,True,True
TLT,True,False,True,True


Table shows the correct alignment of targets and input rows.

# Define Targets

We will experiment with both regression - predicting target next day returns, and classification - predicting target next day direction.

We will treat them as separate experiements building and testing separate models for each. We may evantually conclude that predicting one target works better compared to predicting the other.

# Define Modelling Features

We will establish a baseline based on a delibirately reasonable feature set.

```
daily_return
lag_1_return
lag_5_return
rolling_7d_return
rolling_30d_return
price_vs_ma20
moving_avg_20
moving_avg_50
rolling_30d_volatility
drawdown
relative_volume
```

It is important here that we do not include target vairables, future values, duplicate versions of the same information (e.g., simple and log daily return) unless we want to delibirately test them, other calendar versions.

We will then evaluate whether feature reduction, addition or alternative combinations will help the model prediction.

In [223]:
df_2 = df.copy()

all_cols = df_2.columns

generated_cols = [
    col for col in all_cols
    if col.endswith("_no_calendar")
]

baseline_features = ['daily_return_no_calendar', 'lag_1_return_no_calendar',
       'lag_5_return_no_calendar', 'rolling_7d_return_no_calendar',
       'rolling_30d_return_no_calendar', 'moving_avg_20_no_calendar',
       'moving_avg_50_no_calendar', 'price_vs_ma20_no_calendar',
       'relative_volume_no_calendar', 'rolling_30d_volatility_no_calendar',
       'drawdown_no_calendar']

target_features = ['target_next_day_return_no_calendar', 'target_direction_no_calendar']

additional_features = [
    col for col in generated_cols
    if col not in baseline_features and col not in target_features
]

base_cols = [
    col for col in all_cols
    if col not in generated_cols
]

all_sets = baseline_features + target_features + additional_features + base_cols

set_check = (
    len(all_sets) == len(set(all_sets))
    and set(all_sets) == set(all_cols)
)

print("Baseline features:")
print(baseline_features)

print("\n")
print("Target features:")
print(target_features)

print("\n")
print("Additional features:")
print(additional_features)

print("\n")
print("Original columns:")
print(base_cols)

print("\n")
print("Set Check:")
print(set_check)



Baseline features:
['daily_return_no_calendar', 'lag_1_return_no_calendar', 'lag_5_return_no_calendar', 'rolling_7d_return_no_calendar', 'rolling_30d_return_no_calendar', 'moving_avg_20_no_calendar', 'moving_avg_50_no_calendar', 'price_vs_ma20_no_calendar', 'relative_volume_no_calendar', 'rolling_30d_volatility_no_calendar', 'drawdown_no_calendar']


Target features:
['target_next_day_return_no_calendar', 'target_direction_no_calendar']


Additional features:
['log_return_no_calendar', 'cumulative_returns_no_calendar']


Original columns:
['ticker', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume']


Set Check:
True


# Create Chronological Splits

We should never randomly split time series data - the model is always going to be trained on past data and evaluated on future observations. The requirement is that train < validation < test.

It is important that we do not force the same number of observations within each set. For later IPO assets like SNDK and RPI.L we shoudl split by percent within each ticker. When building a pooled model we can determine how to deal with differing start dates.

One thing that is worrying is the recent market trends and changes that have diverted from past behaviours.

In [224]:
# Split in proportions by ticker

df_3 = df.copy()
ticker = "ticker"

train_sets = []
validation_sets = []
test_sets = []

for asset, group in df_3.groupby(ticker):

    group = group.sort_values("date").reset_index(drop=True)

    n = len(group)

    train_end = int(n * 0.7)
    validation_end = int(n * 0.85)

    train_sets.append(group.iloc[:train_end])
    validation_sets.append(group.iloc[train_end:validation_end])
    test_sets.append(group.iloc[validation_end:])

train_ticker = pd.concat(train_sets, ignore_index=True)
validation_ticker = pd.concat(validation_sets, ignore_index=True)
test_ticker = pd.concat(test_sets, ignore_index=True)

data_split_summary = pd.concat([
    train_ticker.assign(split="Train"),
    validation_ticker.assign(split="Validation"),
    test_ticker.assign(split="Test")
])

data_split_summary = (
    data_split_summary
    .groupby([ticker, "split"])
    .agg(
        rows=("date", "size"),
        start_date=("date","min"),
        end_date=("date","max")
    ).reset_index()
)

display(data_split_summary.style.hide(axis="index"))

ticker,split,rows,start_date,end_date
GLD,Test,324,2025-04-16 00:00:00,2026-07-31 00:00:00
GLD,Train,1509,2018-01-02 00:00:00,2023-12-29 00:00:00
GLD,Validation,323,2024-01-02 00:00:00,2025-04-15 00:00:00
MU,Test,324,2025-04-16 00:00:00,2026-07-31 00:00:00
MU,Train,1509,2018-01-02 00:00:00,2023-12-29 00:00:00
MU,Validation,323,2024-01-02 00:00:00,2025-04-15 00:00:00
NKE,Test,324,2025-04-16 00:00:00,2026-07-31 00:00:00
NKE,Train,1509,2018-01-02 00:00:00,2023-12-29 00:00:00
NKE,Validation,323,2024-01-02 00:00:00,2025-04-15 00:00:00
RPI.L,Test,82,2026-04-07 00:00:00,2026-07-31 00:00:00


We will train models based on individual ticker splits using the 70:15:15 train:validation:test split. I am slightly worried since the test set captures all recent data and so is likely to not match the behvaiour seen in the past.

In [225]:
# Pooled chronological splits

df_4 = df.copy()
date = "date"

unique_dates = np.sort(df_4[date].unique())

train_end = int(len(unique_dates) * 0.7)
validation_end = int(len(unique_dates) * 0.85)

train_end_date = unique_dates[train_end]
validation_end_date = unique_dates[validation_end]

pooled_train = df_4[df_4[date] < train_end_date].copy()

pooled_validation = df_4[
    (df_4[date] >= train_end_date) &
    (df_4[date] < validation_end_date)
].copy()

pooled_test = df_4[df_4[date] >= validation_end_date].copy()

pooled_split_summary = pd.DataFrame({
    "split": ["Train", "Validation", "Test"],
    "rows": [
        len(pooled_train),
        len(pooled_validation),
        len(pooled_test)
    ],
    "start_date": [
        pooled_train["date"].min(),
        pooled_validation["date"].min(),
        pooled_test["date"].min()
    ],
    "end_date": [
        pooled_train["date"].max(),
        pooled_validation["date"].max(),
        pooled_test["date"].max()
    ]
})

display(pooled_split_summary.style.hide(axis="index"))

split,rows,start_date,end_date
Train,7595,2018-01-02 00:00:00,2024-01-16 00:00:00
Validation,1866,2024-01-17 00:00:00,2025-04-24 00:00:00
Test,2228,2025-04-25 00:00:00,2026-07-31 00:00:00


For now we used manual chronological slicing for the train/validation/test sets but we will potentially use TimeSeriesSplit later if we decide to do cross validation or hyperparameter tuning.

With respect to shifting regimes, especially in more recent times, we may perform a walk-forward / expanding window validation alongside the final holdout test. This may look something like:

```
Fold 1
TRAIN:  2018 ───────── 2021
VAL:                       2022

Fold 2
TRAIN:  2018 ───────────────── 2022
VAL:                              2023

Fold 3
TRAIN:  2018 ─────────────────────── 2023
VAL:                                  2024

Fold 4
TRAIN:  2018 ───────────────────────────── 2024
VAL:                                        2025

FINAL TEST:                                  2025/26 ──>
```

However, due to the use of our features a shift in regimes doesn't necessarily mean that the older data is useless. For example, training using features means that when moving average is below a certtain threshold while volume is above a threshold we are more likely to see a price increase.

# Calculate Trivial Baselines

Regression baselines:
- always predicting 0.
- always predicting the training sets mean next day return.

Regression evaluated with:
- MAE
- RMSE
- R^2

Classification baselines:
- always predict the majority training class.

Classification evaluated with:
- accuracy
- precision
- recall
- F1

We apply these baselines to tickers individually then on the pooled set.

When coming to the actual modelling in later notebooks we will have an additional column that indicates the difference.

We will first create baselines on the validation sets and use those as comparisons until final evaluation when we will recreate the baselines on the test sets.

In [226]:
# Always predicting zeros

from sklearn.metrics import (mean_absolute_error,
                            root_mean_squared_error,
                            r2_score)

val_set1 = validation_ticker.copy()
ticker = "ticker"
target = "target_next_day_return_no_calendar"

zero_results = []

for asset, group in val_set1.groupby(ticker):

    val_length = len(group)
    y_pred = np.zeros(val_length)
    y_actual = group[target]

    mae = mean_absolute_error(y_actual, y_pred)*100
    rmse = root_mean_squared_error(y_actual, y_pred)*100
    r2_s = r2_score(y_actual, y_pred)

    zero_result = pd.DataFrame({
        "Data": [asset],
        "MAE (%)": [mae],
        "RMSE (%)": [rmse],
        "R^2": [r2_s]
    })

    zero_results.append(zero_result)

val_length = len(pooled_validation)
y_pred = np.zeros(val_length)
y_actual = pooled_validation[target]

mae = mean_absolute_error(y_actual, y_pred)*100
rmse = root_mean_squared_error(y_actual, y_pred)*100
r2_s = r2_score(y_actual, y_pred)

zero_result = pd.DataFrame({
        "Data": "Pooled",
        "MAE (%)": [mae],
        "RMSE (%)": [rmse],
        "R^2": [r2_s]
    })

zero_results.append(zero_result)

zero_pred_results = pd.concat(zero_results)

print("Baseline: Alwasy predicting 0")
display(zero_pred_results.style.hide(axis="index"))


Baseline: Alwasy predicting 0


Data,MAE (%),RMSE (%),R^2
GLD,0.759164,0.991006,-0.024339
MU,2.670843,3.752859,-0.000026
NKE,1.392219,2.321591,-0.005910
RPI.L,4.156743,8.039907,-0.007859
SNDK,5.287747,6.439544,-0.067260
SPY,0.714254,1.143330,-0.001501
TLT,0.716577,0.901096,-0.000224
Pooled,1.532083,2.651360,-0.000220


## MAE:

MAE represents the average absolute size of prediction error.

Taking GLD as an example, an MAE = 0.71% suggests that if we predict 0% return everyday our prediction is off by 0.71% on average.

Comparing MAE across tickers can be misleading as certain assets may have larger movements and so their prediction errors will naturally tend to be larger.

It is therefore important to only compare within the same data. We will investigate whether our model reduces the MAE below the baseline for the given data.

## RMSE: 

RMSE also measures prediction error, but it punishes larger errors more strongly.

Take RPI.L with MAE = 4.16% and RMSE = 8.04% the large difference represent there are likely observations where returns are very far from zero. Those large errors pull RMSE upwards more strongly than MAE.

## R^2 Score:

R2 score is slightly different. A score of 1 indicates perfect prediction, a score of 0 is an approximate equivalent performance to predicting the validation set mean and negative is worse than predicting the validation set mean.

In [227]:
# Always predicitng training mean

val_set2 = validation_ticker.copy()
train_set2 = train_ticker.copy()
ticker = "ticker"
target = "target_next_day_return_no_calendar"

mean_results = []

for (asset_train, group_train),(asset_val, group_val) in zip(train_set2.groupby(ticker), val_set2.groupby(ticker)):

    val_len = len(group_val)
    y_pred = np.full(val_len, group_train[target].mean())
    y_actual = group_val[target]

    mae = mean_absolute_error(y_actual, y_pred)*100
    rmse = root_mean_squared_error(y_actual, y_pred)*100
    r2_s = r2_score(y_actual, y_pred)

    mean_result = pd.DataFrame({
        "Data": [asset_train],
        "MAE (%)": [mae],
        "RMSE (%)": [rmse],
        "R^2": [r2_s]
    })

    mean_results.append(mean_result)

pool_val = pooled_validation
pool_train = pooled_train

val_len = len(pool_val)
y_pred = np.full(val_len, pool_train[target].mean())
y_actual = pool_val[target]

mae = mean_absolute_error(y_actual, y_pred)*100
rmse = root_mean_squared_error(y_actual, y_pred)*100
r2_s = r2_score(y_actual, y_pred)

mean_result = pd.DataFrame({
    "Data": "Pooled",
    "MAE (%)": [mae],
    "RMSE (%)": [rmse],
    "R^2": [r2_s]
})

mean_results.append(mean_result)

mean_pred_results = pd.concat(mean_results)

print("Baseline: Always predicting the training mean")
display(mean_pred_results.style.hide(axis="index"))


Baseline: Always predicting the training mean


Data,MAE (%),RMSE (%),R^2
GLD,0.754380,0.986590,-0.015230
MU,2.669519,3.753396,-0.000312
NKE,1.395395,2.326882,-0.010500
RPI.L,4.158453,8.039192,-0.007679
SNDK,5.249340,6.239299,-0.001916
SPY,0.708455,1.142502,-0.000050
TLT,0.716613,0.901073,-0.000173
Pooled,1.530094,2.651072,-0.000003


In [228]:
diffs = (zero_pred_results[["MAE (%)", "RMSE (%)", "R^2"]] - 
        mean_pred_results[["MAE (%)", "RMSE (%)", "R^2"]])

diffs.insert(0, "Data", zero_pred_results["Data"])


print("Zero prediction results minus mean prediction results")
print("+ve MAE indicates better training, \n+ve RMSE indicates better training, \n-ve r2 indicates better training.")
display(diffs.style.hide(axis="index"))

Zero prediction results minus mean prediction results
+ve MAE indicates better training, 
+ve RMSE indicates better training, 
-ve r2 indicates better training.


Data,MAE (%),RMSE (%),R^2
GLD,0.004784,0.004416,-0.009109
MU,0.001324,-0.000537,0.000286
NKE,-0.003176,-0.005291,0.004590
RPI.L,-0.001710,0.000715,-0.000179
SNDK,0.038407,0.200246,-0.065344
SPY,0.005799,0.000828,-0.001451
TLT,-0.000036,0.000023,-0.000051
Pooled,0.001989,0.000288,-0.000217


## Comparing 0 prediction and train-mean baselines:

### GLD:

Predicting training mean performed slightly better than 0 with MAE and RMSE, while r2 increased. This suggests training mean is better at constant prediction, but more negative r2 shows that both constant baselines still perform worse in terms of squared error than predicting validation set mean.

### MU:

Two baselines have almost identical results. Trainign mean slightly reduced MAE while zero had marginally lower RMSE and hgiher r2. Suggests training mean slightly improved absolute errors, but predicting 0 was better for large errors. The differences are extremely small though.

### NKE:

0 prediction performed slightly better across all three metrics. For 0 baselines both MAE and RMSE were lower and r2 was higher indicating constant 0 was slightly better prediction than historical training mean.

### RPI.L:

Small marginal difference with 0 prediction having lower MAE but higher RMSE, and more negative r2. 

### SNDK:

Training mean constant had lower MAE and RMSE and lower r2. Substamtial improvement of r2 indicating trianing mean closer to validation mean prediction performance.

### SPY:

Predicting training mean performed better than 0 prediction. Improvement is small though.

### TLT:

Two baselines are essentially equivalent with negligible differences.


### Pooled:

Training mean had better performance than 0 prediction across all three metrics.

## Overall conclusion:

Across most assets, predicting zero and predicting the training mean produce very similar results. This is consistent with daily returns having relatively small average values compared with their day-to-day volatility.

For GLD, SPY and the pooled dataset, the training mean performs slightly better. For NKE, zero performs slightly better. MU, RPI.L and TLT show almost no meaningful difference between the two baselines.

SNDK is the main exception, where predicting the training mean produces a noticeably better RMSE and R² than predicting zero.

These baselines therefore provide a useful minimum benchmark: any trained regression model should ideally achieve lower MAE and RMSE and a higher R² than both simple constant-prediction approaches.



In [229]:
from sklearn.metrics import (accuracy_score,
                             precision_score,
                             recall_score,
                             f1_score)

def classification_results(data, training_df, validation_df):

    target = "target_direction_no_calendar"

    y_actual = validation_df[target]

    y_len = len(validation_df)
    y_pred = np.full(y_len, training_df[target].mode())

    accuracy = accuracy_score(y_actual, y_pred)*100
    precision = precision_score(y_actual, y_pred)*100
    recall = recall_score(y_actual, y_pred)*100
    f1 = f1_score(y_actual, y_pred)*100

    result = {
        "Data": data,
        "Prediction": training_df[target].mode().iloc[0],
        "Accuracy (%)": accuracy,
        "Precision (%)": precision,
        "Recall (%)": recall,
        "F1 Score (%)": f1 
    }

    return result

ticker = "ticker"

train_set3 = train_ticker.copy()
val_set3 = validation_ticker.copy()
pool_train3 = pooled_train.copy()
pool_val3 = pooled_validation.copy()

results = []

for (asset_train, group_train), (asset_val, group_val) in zip(train_set3.groupby(ticker), val_set3.groupby(ticker)):

    result = classification_results(asset_train, group_train, group_val)

    results.append(result)

result = classification_results("Pooled", pool_train3, pool_val3)

results.append(result)

result_table = pd.DataFrame(results)

display(result_table.style.hide(axis="index"))

/opt/miniconda3/envs/market-intelligence-pipeline/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Data,Prediction,Accuracy (%),Precision (%),Recall (%),F1 Score (%)
GLD,1,58.513932,58.513932,100.000000,73.828125
MU,1,51.083591,51.083591,100.000000,67.622951
NKE,1,47.987616,47.987616,100.000000,64.853556
RPI.L,0,60.493827,0.000000,0.000000,0.000000
SNDK,1,56.363636,56.363636,100.000000,72.093023
SPY,1,57.275542,57.275542,100.000000,72.834646
TLT,1,50.773994,50.773994,100.000000,67.351129
Pooled,1,52.840300,52.840300,100.000000,69.144460


## Metrics:

|                        | Actually Up (`1`)       | Actually Down (`0`)     |
| ---------------------- | ----------------------- | ----------------------- |
| **Predict Up (`1`)**   | **True Positive (TP)**  | **False Positive (FP)** |
| **Predict Down (`0`)** | **False Negative (FN)** | **True Negative (TN)**  |


Accuracy: 

> of all prediction how many were correct?

Precision: 

> when predicting 1 how many predicted 1 were correct.

Recall: 

> of all the 1 days, how many were found.

F1: 

> combines precision and recall, evaluating the balance between precision and recall, with a higher score meaning better balance.

## Results:

Precision, recall and F1 are reported fro completeness, but are of limited interpretive value for a constant class baseline because the classifier only predicts one class. This mechanically produces either 100% or 0% recall for the positive class rather than demonstrating genuine classification ability.

This baseline predicts the most common class from the training data for every validation observation. For most assets the mode is 1 with RPI.L as the exception.

GLD has the strongest majority class baseline accuracy and the pooled dataset is close to 50% indicating that the validation classes are relatively balanced and that predicting majority training class has little advantage over chance level directional classification.

NKE should be noted as training class majority is 1 while only 48% of validation observations are correctly predicted. This shows the dominant class changes between training and validation periods.

RPI.L behaves differently because the major class is 0. The baseline predicts down and achieves 60.5%. accuracy as down is the majority in the validation.

Overall, class imbalance alone can produce accuracy around 50-60% without any genuine predictive ability. 100% recall and high f1 are artefacts of always predictign 1 rather than strong classification performance. 

These results simply provide a minimum benchmark that trained classifiers should improve upon, ideally by achieving better accuracy while successfully identifying both up and down observations.